In [1]:
import os
import ray
import numpy as np
import pandas as pd
from typing import List
from utils.data import load_yaml
from parsers.surface_track_id_parser import SurfaceObjectTrackParserDistributed
from imaris.imaris import ImarisDataObject

In [2]:
config_path = "config/config.yaml"
config = load_yaml(config_path)
config

{'data_dir': ['/home/shehan/Documents/projects/nih/nih_parsers/data/temp_test'],
 'save_dir': ['/home/shehan/Documents/projects/nih/nih_parsers/data/temp_test'],
 'valid_surfaces': -1,
 'cpu_cores': 12}

In [3]:
ims_file = "/home/shehan/Documents/projects/nih/nih_parsers/data/temp_test/Ex112-mitotrackerred-B6-BALB-050125_batch_TileScan_1_BALB-US-Position_3_(0).ims"

In [4]:
ims_object = ImarisDataObject(ims_file)
data = ims_object.data

In [5]:
data["Scene8"]["Content"]["MegaSurfaces0"].keys()

<KeysViewHDF5 ['BlockData', 'BlockInfo', 'BlockPath', 'Category', 'CategoryFunction', 'CreationParameters', 'Factor', 'FactorFunction', 'FactorList', 'FactorListFunction', 'LabelColor', 'LabelColorData', 'LabelColorLabelGroupNames', 'LabelColorLabelValues', 'LabelGroupNames', 'LabelSetLabelIDs', 'LabelSetObjectIDs', 'LabelSets', 'LabelValues', 'LevelInfo', 'MainTrackSegmentTable', 'MainTrackSegmentTable_Focus', 'MainTrackTable', 'SplitOffset', 'StatisticsType', 'StatisticsTypeFunction', 'StatisticsValue', 'StatisticsValueFunction', 'StatisticsValueTimeOffset', 'StatisticsValueTimeOffsetFunction', 'SurfaceModel', 'SurfaceModel2DInfo', 'SurfaceModelInfo', 'SurfaceTimeOffset', 'Time', 'TimeBegin', 'Track0', 'TrackEdge0', 'TrackObject0', 'TrackSegment0', 'TrackSegment0_Focus']>

In [6]:
parser = SurfaceObjectTrackParserDistributed(ims_file)

In [7]:
out = parser.inspect(0)

In [9]:
print(f"all surface names: {parser.surface_names}")
print(f"surface name: {out["surface_name"]}")
print(list(out.keys()))

all surface names: ['MegaSurfaces0']
surface name: MegaSurfaces0
['surface_name', 'stat_names_raw', 'stat_values_raw', 'object_id', 'factor', 'stat_names_channel_info', 'stat_names_surface_info', 'stat_values_filtered', 'organized_stats', 'stats_df', 'final_df']


In [16]:
stats_df = out["stats_df"]
database = out["final_df"]
track_info = parser.track_info.get(0)
object_info = parser.object_info.get(0)

In [17]:
database

,Acceleration,Acceleration X,Acceleration Y,Acceleration Z,Area,BoundingBoxAA Length X,BoundingBoxAA Length Y,BoundingBoxAA Length Z,BoundingBoxOO Length A,BoundingBoxOO Length B,...,Time Since Track Start,Velocity Angle X,Velocity Angle Y,Velocity Angle Z,Velocity X,Velocity Y,Velocity Z,Volume,Object_ID,Track_ID
0,0.000000,0.000000,0.000000,0.000000,2612.532959,41.707489,31.091003,15.998888,14.792021,30.354095,...,0.0,115.730408,73.730881,148.896225,-0.136261,0.087952,-0.268763,6740.480957,0,1.000000e+09
1,0.000000,0.000000,0.000000,0.000000,614.430847,15.166351,14.407990,13.999027,14.607727,15.557465,...,0.0,73.654175,17.273527,84.577354,1.554016,5.272064,0.521990,1032.898804,1,1.000000e+09
2,NaN,NaN,NaN,NaN,595.933167,14.408051,15.166306,13.999027,13.209564,13.857574,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1083.411377,2,NaN
3,0.000000,0.000000,0.000000,0.000000,909.628601,13.649719,21.232864,13.999027,13.293793,14.529671,...,0.0,52.122349,38.067715,86.773430,2.964966,3.801804,0.272010,2185.329102,3,1.000000e+09
4,0.000000,0.000000,0.000000,0.000000,603.822021,13.649719,13.649704,13.999027,12.407135,13.171967,...,0.0,58.979187,129.567520,124.986053,0.738525,-0.912720,-0.821565,1312.139893,4,1.000000e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32440,8.411903,-0.619156,-8.360657,0.690062,3.143441,3.033279,2.274948,3.999722,0.991871,1.032562,...,1.0,138.958405,130.605240,84.977478,-1.002892,-0.865402,0.116475,0.281079,32440,1.000028e+09
32441,14.271921,2.916595,-10.283104,9.457218,16.039974,3.033264,6.066536,3.999722,1.663489,2.476746,...,2.0,115.543518,100.650925,27.974693,-0.155563,-0.066669,0.318659,2.889613,32441,1.000030e+09
32442,10.697466,-5.497772,-5.976265,6.963802,1.417993,3.033264,2.274948,3.999722,0.569454,0.655510,...,4.0,138.119354,76.449181,128.697266,-1.864182,0.586796,-1.565367,0.097615,32442,1.000030e+09
32443,2.748623,-1.244181,-1.651446,1.810985,2.237356,2.274963,3.791588,3.999722,0.187986,0.939240,...,6.0,15.466559,74.690781,92.132759,2.736959,0.749891,-0.105548,0.092679,32443,1.000030e+09


In [18]:
# how many unique object ids
print(len(stats_df["Object_ID"].unique()))
print(len(object_info["ID_Object"].unique()))

31427
31075


In [19]:
track_info

,ID,IndexTrackObjectBegin,IndexTrackObjectEnd,IndexTrackEdgeBegin,IndexTrackEdgeEnd
0,1000000000,0,51,0,50
1,1000000001,51,155,50,153
2,1000000003,155,247,153,244
3,1000000004,247,361,244,357
4,1000000005,361,480,357,475
...,...,...,...,...,...
606,1000031677,31062,31064,30456,30457
607,1000031715,31064,31067,30457,30459
608,1000031744,31067,31070,30459,30461
609,1000031815,31070,31073,30461,30463


In [20]:
object_info[13]

KeyError: 13

In [34]:
rev_object_info = {v:k for k,v in object_info.to_dict()["ID_Object"].items()}

In [37]:
rev_object_info

{0: 0,
 351: 1,
 547: 2,
 808: 3,
 1074: 4,
 1338: 5,
 1610: 6,
 1893: 7,
 2156: 8,
 2431: 9,
 2697: 10,
 2980: 11,
 3325: 12,
 3588: 13,
 3789: 14,
 4057: 15,
 4328: 16,
 4607: 17,
 4881: 18,
 5166: 19,
 5509: 20,
 5704: 21,
 5972: 22,
 6249: 23,
 6526: 24,
 6805: 25,
 7078: 26,
 7342: 27,
 7624: 28,
 7901: 29,
 8253: 30,
 8450: 31,
 8798: 32,
 8999: 33,
 9357: 34,
 9626: 35,
 9899: 36,
 10097: 37,
 10386: 38,
 10667: 39,
 11005: 40,
 11214: 41,
 11486: 42,
 11760: 43,
 12039: 44,
 12583: 45,
 12857: 46,
 13126: 47,
 13393: 48,
 13658: 49,
 13919: 50,
 1: 51,
 265: 52,
 546: 53,
 809: 54,
 1075: 55,
 1340: 56,
 1609: 57,
 1880: 58,
 2155: 59,
 2435: 60,
 2780: 61,
 2976: 62,
 3326: 63,
 3587: 64,
 3787: 65,
 4058: 66,
 4326: 67,
 4603: 68,
 4878: 69,
 5162: 70,
 5427: 71,
 6050: 72,
 6244: 73,
 6522: 74,
 7344: 75,
 8178: 76,
 8453: 77,
 8720: 78,
 9280: 79,
 9559: 80,
 9829: 81,
 10100: 82,
 10379: 83,
 10656: 84,
 10936: 85,
 11213: 86,
 11556: 87,
 11750: 88,
 12029: 89,
 12304: 90

In [20]:
stats_df["Track_ID"] = stats_df.apply(
    func=lambda x: database[int(x["Object_ID"].item())],
    axis=1,
)


KeyError: 2

In [23]:
out["final_df"]

{np.int64(0): np.int64(1000000000),
 np.int64(351): np.int64(1000000000),
 np.int64(547): np.int64(1000000000),
 np.int64(808): np.int64(1000000000),
 np.int64(1074): np.int64(1000000000),
 np.int64(1338): np.int64(1000000000),
 np.int64(1610): np.int64(1000000000),
 np.int64(1893): np.int64(1000000000),
 np.int64(2156): np.int64(1000000000),
 np.int64(2431): np.int64(1000000000),
 np.int64(2697): np.int64(1000000000),
 np.int64(2980): np.int64(1000000000),
 np.int64(3325): np.int64(1000000000),
 np.int64(3588): np.int64(1000000000),
 np.int64(3789): np.int64(1000000000),
 np.int64(4057): np.int64(1000000000),
 np.int64(4328): np.int64(1000000000),
 np.int64(4607): np.int64(1000000000),
 np.int64(4881): np.int64(1000000000),
 np.int64(5166): np.int64(1000000000),
 np.int64(5509): np.int64(1000000000),
 np.int64(5704): np.int64(1000000000),
 np.int64(5972): np.int64(1000000000),
 np.int64(6249): np.int64(1000000000),
 np.int64(6526): np.int64(1000000000),
 np.int64(6805): np.int64(10000

In [35]:
sorted(object_info["ID_Object"].unique())

[np.int64(0),
 np.int64(1),
 np.int64(3),
 np.int64(4),
 np.int64(5),
 np.int64(6),
 np.int64(7),
 np.int64(8),
 np.int64(9),
 np.int64(10),
 np.int64(11),
 np.int64(12),
 np.int64(13),
 np.int64(14),
 np.int64(15),
 np.int64(16),
 np.int64(17),
 np.int64(18),
 np.int64(19),
 np.int64(20),
 np.int64(21),
 np.int64(22),
 np.int64(23),
 np.int64(24),
 np.int64(25),
 np.int64(26),
 np.int64(29),
 np.int64(30),
 np.int64(31),
 np.int64(33),
 np.int64(34),
 np.int64(35),
 np.int64(36),
 np.int64(37),
 np.int64(38),
 np.int64(39),
 np.int64(40),
 np.int64(41),
 np.int64(42),
 np.int64(43),
 np.int64(44),
 np.int64(45),
 np.int64(47),
 np.int64(48),
 np.int64(49),
 np.int64(50),
 np.int64(51),
 np.int64(53),
 np.int64(54),
 np.int64(55),
 np.int64(56),
 np.int64(57),
 np.int64(58),
 np.int64(59),
 np.int64(60),
 np.int64(61),
 np.int64(62),
 np.int64(63),
 np.int64(64),
 np.int64(65),
 np.int64(66),
 np.int64(67),
 np.int64(68),
 np.int64(70),
 np.int64(71),
 np.int64(72),
 np.int64(73),
 np.

,ID,IndexTrackObjectBegin,IndexTrackObjectEnd,IndexTrackEdgeBegin,IndexTrackEdgeEnd
0,1000000000,0,51,0,50
1,1000000001,51,155,50,153
2,1000000003,155,247,153,244
3,1000000004,247,361,244,357
4,1000000005,361,480,357,475
...,...,...,...,...,...
606,1000031677,31062,31064,30456,30457
607,1000031715,31064,31067,30457,30459
608,1000031744,31067,31070,30459,30461
609,1000031815,31070,31073,30461,30463


In [ ]:
sorted(object_info["ID_Object"].unique())

[np.int64(0),
 np.int64(1),
 np.int64(3),
 np.int64(4),
 np.int64(5),
 np.int64(6),
 np.int64(7),
 np.int64(8),
 np.int64(9),
 np.int64(10),
 np.int64(11),
 np.int64(12),
 np.int64(13),
 np.int64(14),
 np.int64(15),
 np.int64(16),
 np.int64(17),
 np.int64(18),
 np.int64(19),
 np.int64(20),
 np.int64(21),
 np.int64(22),
 np.int64(23),
 np.int64(24),
 np.int64(25),
 np.int64(26),
 np.int64(29),
 np.int64(30),
 np.int64(31),
 np.int64(33),
 np.int64(34),
 np.int64(35),
 np.int64(36),
 np.int64(37),
 np.int64(38),
 np.int64(39),
 np.int64(40),
 np.int64(41),
 np.int64(42),
 np.int64(43),
 np.int64(44),
 np.int64(45),
 np.int64(47),
 np.int64(48),
 np.int64(49),
 np.int64(50),
 np.int64(51),
 np.int64(53),
 np.int64(54),
 np.int64(55),
 np.int64(56),
 np.int64(57),
 np.int64(58),
 np.int64(59),
 np.int64(60),
 np.int64(61),
 np.int64(62),
 np.int64(63),
 np.int64(64),
 np.int64(65),
 np.int64(66),
 np.int64(67),
 np.int64(68),
 np.int64(70),
 np.int64(71),
 np.int64(72),
 np.int64(73),
 np.

In [39]:
object_info

,ID_Object
0,0
1,351
2,547
3,808
4,1074
...,...
31070,31815
31071,32081
31072,32346
31073,31961


In [46]:
sorted(object_info[0:51]['ID_Object'])

[0,
 351,
 547,
 808,
 1074,
 1338,
 1610,
 1893,
 2156,
 2431,
 2697,
 2980,
 3325,
 3588,
 3789,
 4057,
 4328,
 4607,
 4881,
 5166,
 5509,
 5704,
 5972,
 6249,
 6526,
 6805,
 7078,
 7342,
 7624,
 7901,
 8253,
 8450,
 8798,
 8999,
 9357,
 9626,
 9899,
 10097,
 10386,
 10667,
 11005,
 11214,
 11486,
 11760,
 12039,
 12583,
 12857,
 13126,
 13393,
 13658,
 13919]